# 02 - Preprocessing

_Placeholder notebook. Add content later._


In [1]:
import pandas as pd
import numpy as np
import nltk 

# Data Exploration

In [2]:
# loading the data
df = pd.read_csv('WELFake_Dataset.csv')


In [3]:
# printing summary of the data
print(df.info()) # information abouth the data
print(df.head(5)) # print first 5 rows
print(df.columns.to_list()) # showing the data fram columns
print(df.describe())

<class 'pandas.DataFrame'>
RangeIndex: 72134 entries, 0 to 72133
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   Unnamed: 0  72134 non-null  int64
 1   title       71576 non-null  str  
 2   text        72095 non-null  str  
 3   label       72134 non-null  int64
dtypes: int64(2), str(2)
memory usage: 2.2 MB
None
   Unnamed: 0                                              title  \
0           0  LAW ENFORCEMENT ON HIGH ALERT Following Threat...   
1           1                                                NaN   
2           2  UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...   
3           3  Bobby Jindal, raised Hindu, uses story of Chri...   
4           4  SATAN 2: Russia unvelis an image of its terrif...   

                                                text  label  
0  No comment is expected from Barack Obama Membe...      1  
1     Did they post their votes for Hillary already?      1  
2   Now, most of the de

In [4]:
df_clean = df.rename(columns={'Unnamed: 0': 'index'})  # changing the noun of unamed : 0 to index of the news
df_clean.columns.to_list()

['index', 'title', 'text', 'label']

In [5]:
print(df.describe())
print(df['label'].value_counts())
print(df.isnull().sum())

         Unnamed: 0         label
count  72134.000000  72134.000000
mean   36066.500000      0.514404
std    20823.436496      0.499796
min        0.000000      0.000000
25%    18033.250000      0.000000
50%    36066.500000      1.000000
75%    54099.750000      1.000000
max    72133.000000      1.000000
label
1    37106
0    35028
Name: count, dtype: int64
Unnamed: 0      0
title         558
text           39
label           0
dtype: int64


In [6]:
# replace null cases with space
df_clean['text'] = df_clean['text'].fillna('')
df_clean['title'] = df_clean['title'].fillna('')
print(df_clean.isnull().sum()) # verification

index    0
title    0
text     0
label    0
dtype: int64


In [7]:
# conctatenation of the column title et texte
df_clean['news_conteant'] = df_clean['title'].str.cat(df_clean['text'],sep = ',')
print(df_clean.columns.to_list())

['index', 'title', 'text', 'label', 'news_conteant']


In [8]:
df_clean.drop(columns=['title'],inplace=True)
df_clean.drop(columns=['text'],inplace=True)
print(df_clean.columns.tolist())

['index', 'label', 'news_conteant']


In [9]:
print(df_clean)


       index  label                                      news_conteant
0          0      1  LAW ENFORCEMENT ON HIGH ALERT Following Threat...
1          1      1    ,Did they post their votes for Hillary already?
2          2      1  UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...
3          3      0  Bobby Jindal, raised Hindu, uses story of Chri...
4          4      1  SATAN 2: Russia unvelis an image of its terrif...
...      ...    ...                                                ...
72129  72129      0  Russians steal research on Trump in hack of U....
72130  72130      1   WATCH: Giuliani Demands That Democrats Apolog...
72131  72131      0  Migrants Refuse To Leave Train At Refugee Camp...
72132  72132      0  Trump tussle gives unpopular Mexican leader mu...
72133  72133      1  Goldman Sachs Endorses Hillary Clinton For Pre...

[72134 rows x 3 columns]


# Texte cleaning 

In [21]:
import re
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize # tokenazation is nedeed befor stemming
stop_words = set(stopwords.words('english')) # selection of stop words to remove
def removing_stop_words_and_stemming(text):
    stemmer = PorterStemmer()
    tokenized_text = word_tokenize(text)
    return " ".join(stemmer.stem(m) for m in tokenized_text if m not in stop_words)
        
            
def text_clean(text):
    text = text.lower() 
    text = re.sub(r'^RT[\s]+', '', text) # remove links
    text = re.sub(r'#', '', text) # remove hash 
    text= re.sub(r'[^a-zA-Z\s]', '', text)
    text = removing_stop_words_and_stemming(text)
    return text
  

In [22]:
df_clean['news_content_clean'] = df_clean['news_conteant'].apply(text_clean)

In [ ]:
df_clean = pd.read_csv('cleaned.csv')

In [7]:
# verfiying that cleaning has worked well
print(df_clean[['news_conteant', 'news_content_clean']].sample(5))
print(df.columns)


                                           news_conteant  \
9360   U.S. attorney general unveils 12-city partners...   
64342  Thailand arrests woman wanted over deadly 2015...   
60922  What the WikiLeak Revelations Reveal About Don...   
53506  HERE’S THE BEST WAY To Silence A Liberal Deman...   
9669   VA Backlog Means Thousands Of Veterans Owed Mo...   

                                      news_content_clean  
9360   us attorney gener unveil citi partnership figh...  
64342  thailand arrest woman want deadli bomb shrineb...  
60922  wikileak revel reveal donna brazil dncwhat wik...  
53506  here best way silenc liber demand impeach pres...  
9669   va backlog mean thousand veteran owe money nc ...  
Index(['index', 'label', 'news_conteant', 'news_content_clean'], dtype='str')


In [9]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

# 1. Charger le CSV nettoyé
df = pd.read_csv("cleaned.csv")

# 2. Supprimer les valeurs vides (sécurité)
df = df.dropna()

# 3. Choisir la colonne texte
# (souvent: title, text ou content)
X_text = df["news_content_clean"]   # ou df["text"] ou df["title"]

# 4. Initialiser TF-IDF (tokenizer automatique inclus)
vectorizer = TfidfVectorizer(
    max_features=5000,
    stop_words='english',
    ngram_range=(1,2)
)

# 5. Transformation (Tokenization + TF-IDF)
X = vectorizer.fit_transform(X_text)

# 6. Labels
y = df["label"]

# 7. Vérification
print("Shape des vecteurs :", X.shape)

Shape des vecteurs : (72089, 5000)


In [10]:
from sklearn.model_selection import train_test_split
X_train , X_test , y_train , y_test = train_test_split(
    X, 
    y, 
    test_size=0.2,      # 20% pour le test, 80% pour l'entraînement
    random_state=42,    # Pour que le découpage soit identique à chaque exécution
    stratify=y
)


from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Création du modèle XGBoost
model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss'
)

# Entraînement sur les vecteurs TF-IDF
model.fit(X, y)

# Prédiction
y_pred = model.predict(X_test)

# Évaluation
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

Accuracy: 0.9724649743376335

Classification Report:
               precision    recall  f1-score   support

           0       0.99      0.96      0.97      7006
           1       0.96      0.99      0.97      7412

    accuracy                           0.97     14418
   macro avg       0.97      0.97      0.97     14418
weighted avg       0.97      0.97      0.97     14418


Confusion Matrix:
 [[6703  303]
 [  94 7318]]
